# Monte Carlo Neutronics of a PWR Fuel Pin with OpenMC and pandas
This notebook builds an infinite-lattice PWR UO₂ pin cell (reflective boundaries), solves the k-eigenvalue
problem with OpenMC's continuous-energy Monte Carlo solver, and analyses the tallies systematically with
pandas DataFrames. It is a modernised, heavily annotated counterpart of the official
`openmc-notebooks/pandas-dataframes.ipynb`, updated for OpenMC ≥ 0.16 (Model API, source constraints).

**Learning goals:** the neutron transport equation → Monte Carlo random walks → power iteration and k_eff →
inactive/active batches and Shannon entropy → tallies, estimators and uncertainties → pandas analysis.

## 1. The neutron transport equation
The angular flux ψ(r, Ω, E) (neutrons·cm⁻²·s⁻¹·sr⁻¹·eV⁻¹) obeys the steady-state Boltzmann equation:

$$\Omega\cdot\nabla\psi + \Sigma_t(\mathbf r,E)\psi = \int_0^\infty\!\!\int_{4\pi}\Sigma_s(\mathbf r,E'\to E,\Omega'\to\Omega)\psi\,d\Omega' dE' + \frac{\chi(E)}{4\pi k}\int_0^\infty \nu\Sigma_f(\mathbf r,E')\phi(\mathbf r,E')\,dE'$$

- Σ_x = N σ_x is a macroscopic cross section (cm⁻¹): atom density N (atoms/b-cm) × microscopic cross section σ_x (barns).
- φ = ∫ψ dΩ is the scalar flux, ν is neutrons per fission, χ(E) is the fission spectrum.
- With no external source this is an **eigenvalue problem**: only the factor 1/k makes a steady state possible.
  k_eff > 1 is supercritical, = 1 critical, < 1 subcritical.

Monte Carlo does not discretise this equation. Instead it simulates individual neutron histories whose
*expected* behaviour satisfies it exactly: distance to collision ~ Exp(Σ_t), the collision nuclide is chosen
∝ Σ_{t,i}, and the reaction ∝ σ_x/σ_t, all with continuous-energy nuclear data.